<a href="https://colab.research.google.com/github/rafaelpimentacruz-stack/python_sis_26_2/blob/main/Jorge_Cruz_Aula14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Atividade sala 1
# =====================================================================
# API Escolhida: WeatherAPI (Current Weather API)
# Documentação Oficial: https://www.weatherapi.com/docs/
# =====================================================================

# =====================================================================
# ETAPA 1: Importações (PEP 8), Chave Segura e Lista de 10 Cidades
# =====================================================================
from getpass import getpass

import requests

# Captura segura da API_KEY sem deixá-la exposta no código-fonte
# (Alternativa no Google Colab: from google.colab import userdata; API_KEY = userdata.get('WEATHER_API_KEY'))
API_KEY = getpass("Digite sua chave da WeatherAPI (API_KEY): ").strip()

URL_BASE = "http://api.weatherapi.com/v1/current.json"

cidades_brasileiras = [
    "Curitiba",
    "São Paulo",
    "Rio de Janeiro",
    "Belo Horizonte",
    "Salvador",
    "Fortaleza",
    "Recife",
    "Porto Alegre",
    "Brasília",
    "Manaus",
]


# =====================================================================
# ETAPA 2 e 3: Requisições com Timeout, Tratamento de Erros e Extração JSON
# =====================================================================
resultados_clima = []
chaves_inspecionadas = False

print("\n--- Iniciando Monitoramento Meteorológico ---\n")

for cidade in cidades_brasileiras:
    parametros = {
        "key": API_KEY,
        "q": f"{cidade}, Brazil",
        "lang": "pt",  # Retorna a condição do tempo em português
    }

    try:
        resposta = requests.get(URL_BASE, params=parametros, timeout=10)
        print(f"Consultando '{cidade}' -> Status HTTP: {resposta.status_code}")

        # Dispara HTTPError caso o status seja 4xx (ex: chave/cidade inválida) ou 5xx
        resposta.raise_for_status()

        # Etapa 3: Converte a resposta válida para dicionário Python
        dados_json = resposta.json()

        # Inspeciona as chaves principais do JSON na primeira resposta válida
        if not chaves_inspecionadas:
            print(f"Chaves principais do JSON: {list(dados_json.keys())}\n")
            chaves_inspecionadas = True

        # Armazena o dicionário completo na lista de resultados
        resultados_clima.append(dados_json)

    except requests.exceptions.HTTPError as erro_http:
        print(f"  [Erro HTTP] Falha ao consultar '{cidade}': {erro_http}")
    except requests.exceptions.Timeout:
        print(f"  [Erro Timeout] A requisição para '{cidade}' excedeu 10 segundos.")
    except requests.exceptions.ConnectionError:
        print(f"  [Erro de Conexão] Falha de rede ao tentar acessar a API para '{cidade}'.")
    except requests.exceptions.RequestException as erro_geral:
        print(f"  [Erro Inesperado] Problema na requisição para '{cidade}': {erro_geral}")


# =====================================================================
# EXIBIÇÃO FINAL DOS DADOS COLETADOS
# =====================================================================
print("\n=======================================================")
print("RELATÓRIO DE CLIMA EM TEMPO REAL (CIDADES COLETADAS)")
print("=======================================================")

for registro in resultados_clima:
    nome_cidade = registro["location"]["name"]
    temperatura_c = registro["current"]["temp_c"]
    condicao_tempo = registro["current"]["condition"]["text"]

    print(
        f"Cidade: {nome_cidade:<16} | "
        f"Temperatura: {temperatura_c:>5.1f} °C | "
        f"Condição: {condicao_tempo}"
    )

In [ ]:
#Atividade sala 2
# =====================================================================
# ETAPA 1: Importações (PEP 8), Conversão para DataFrame e Renomeação
# =====================================================================
import matplotlib.pyplot as plt
import pandas as pd

# Verifica se a lista 'resultados_clima' existe na memória do Colab
if "resultados_clima" not in globals() or not resultados_clima:
    raise ValueError(
        "A lista 'resultados_clima' está vazia ou não foi carregada. "
        "Execute primeiro o script da Atividade 01 (WeatherAPI)."
    )

# Achata os dicionários aninhados (location, current, condition) em colunas planas
df_bruto = pd.json_normalize(resultados_clima)

# Seleciona apenas as colunas relevantes para tomada de decisão
colunas_selecionadas = [
    "location.name",
    "location.region",
    "location.localtime",
    "current.temp_c",
    "current.feelslike_c",
    "current.humidity",
    "current.wind_kph",
    "current.condition.text",
]
df = df_bruto[colunas_selecionadas].copy()

# Renomeia para nomes claros em português
df.columns = [
    "Cidade",
    "Estado",
    "Data_Hora_Local",
    "Temperatura_C",
    "Sensacao_Termica_C",
    "Umidade_Pct",
    "Vento_kmh",
    "Condicao_Tempo",
]

print("--- DataFrame Estruturado (Primeiras Linhas) ---")
print(df.head(), "\n")


# =====================================================================
# ETAPA 2: Tratamento, Tipagem de Dados e Resumo Estatístico
# =====================================================================
# 1. Remove eventuais registros duplicados com base na cidade e horário
df = df.drop_duplicates(subset=["Cidade", "Data_Hora_Local"])

# 2. Converte coluna de data/hora para o tipo datetime
df["Data_Hora_Local"] = pd.to_datetime(df["Data_Hora_Local"], errors="coerce")

# 3. Garante tipagem numérica nas métricas meteorológicas
colunas_numericas = [
    "Temperatura_C",
    "Sensacao_Termica_C",
    "Umidade_Pct",
    "Vento_kmh",
]
for col in colunas_numericas:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("--- Verificação dos Tipos de Dados (dtypes) ---")
print(df.dtypes, "\n")

print("--- Resumo Estatístico Consolidado ---")
print(df[colunas_numericas].describe().round(2), "\n")


# =====================================================================
# ETAPA 3: Visualização Comparativa e Exportação Auditável (.xlsx)
# Pergunta: Como a temperatura atual e a sensação térmica se comparam
# entre as capitais monitoradas neste momento?
# =====================================================================
df_plot = df.sort_values(by="Temperatura_C", ascending=True)
media_temp = df_plot["Temperatura_C"].mean()

plt.figure(figsize=(11, 6))

# Barras horizontais para Temperatura Real
barras = plt.barh(
    df_plot["Cidade"],
    df_plot["Temperatura_C"],
    color="#2b5c8f",
    height=0.6,
    label="Temperatura Atual (°C)",
)

# Marcadores de ponto para Sensação Térmica (comparação direta)
plt.scatter(
    df_plot["Sensacao_Termica_C"],
    df_plot["Cidade"],
    color="#e66101",
    s=75,
    zorder=5,
    label="Sensação Térmica (°C)",
)

# Linha vertical tracejada indicando a média geral entre as cidades
plt.axvline(
    media_temp,
    color="#555555",
    linestyle="--",
    linewidth=1.2,
    label=f"Média Nacional ({media_temp:.1f} °C)",
)

# Rótulos de valor na ponta de cada barra
for barra in barras:
    largura = barra.get_width()
    plt.text(
        largura + 0.4,
        barra.get_y() + barra.get_height() / 2,
        f"{largura:.1f}°C",
        va="center",
        ha="left",
        fontsize=9.5,
        fontweight="bold",
        color="#222222",
    )

# Boas práticas visuais: título descritivo, eixos claros e remoção de bordas
plt.title(
    "Monitoramento Climático: Temperatura Atual vs. Sensação Térmica por Cidade",
    fontsize=13,
    fontweight="bold",
    pad=15,
    loc="left",
)
plt.xlabel("Temperatura (°C)", fontsize=11, labelpad=10)
plt.ylabel("Cidade Monitorada", fontsize=11, labelpad=10)
plt.xlim(0, max(df_plot["Temperatura_C"].max(), df_plot["Sensacao_Termica_C"].max()) * 1.18)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.grid(axis="x", linestyle=":", alpha=0.5)
plt.legend(loc="lower right", frameon=True)

plt.tight_layout()
plt.show()

# Exportando o relatório final auditável para Excel (.xlsx) e CSV
arquivo_xlsx = "relatorio_clima_consolidado.xlsx"
df.to_excel(arquivo_xlsx, index=False, sheet_name="Clima_Tempo_Real")

print(f"\nRelatório consolidado exportado com sucesso para '{arquivo_xlsx}'!")